<a href="https://colab.research.google.com/github/kasinadhsarma1/Notebooks/blob/main/Vishwamai_posttrain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# VishwamAI — Stage-1 Post-Training (CWE + Humor) on Kaggle

Continues from the merged stage-0 model. Attaches a **new LoRA** on top of the 16-bit merged weights and fine-tunes on a **CWE/vulnerability dataset** plus a small **humor** mix, then merges to 16-bit and pushes to HF.

**Run order:** 1 → 2 → 3 → 4 (inspect) → 5 (build data) → 6 → 7 → 8 (train) → 9 (save/push) → 10 (test).

Kaggle: Accelerator = **GPU T4**; add **HF_TOKEN** in Add-ons → Secrets.

> Defensive/authorized use only. Train from the MERGED 16-bit model, never the GGUF.

In [1]:
# 1 — install
!pip install -q -U unsloth trl peft datasets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.1/81.1 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.4/25.4 MB 61.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 31.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.3/199.3 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.7/146.7 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 48.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 44.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
# 2 — HF auth (works on Colab AND Kaggle)
import os
from huggingface_hub import login, whoami

HF_TOKEN = None
# Colab secret (key icon in sidebar, name HF_TOKEN, Notebook access ON)
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    pass
# Kaggle secret
if not HF_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
    except Exception:
        pass
# fallback: already-logged-in token (e.g. after notebook_login)
if not HF_TOKEN:
    try:
        from huggingface_hub import get_token
        HF_TOKEN = get_token()
    except Exception:
        pass

if HF_TOKEN:
    login(HF_TOKEN)
else:
    from huggingface_hub import notebook_login
    notebook_login()          # pops a box to paste your token

HF_USER = whoami()['name']
print('logged in as', HF_USER)

logged in as kasinadhsarma


In [3]:
# 3 — config
BASE_MODEL  = 'VishwamAI/vishwamai-merged'   # stage-0 merged (or VishwamAI/vishwamai-merged)
CWE_DATASET   = 'CyberNative/Code_Vulnerability_Security_DPO'  # change to your CWE dataset
CWE_SPLIT     = 'train'
HUMOR_DATASET = 'VishwamAI/Vishwamai_Humours'
HUMOR_SPLIT   = 'train'
OUT_REPO    = f'{HF_USER}/vishwamai-stage1'      # where the new model is pushed
OUT         = '/kaggle/working/stage1'
MAX_SEQ     = 4096
HUMOR_FRAC  = 0.03                                # ~3% humor
MAX_CWE     = 8000                                # cap examples to keep it T4-friendly
SYSTEM = ('You are VishwamAI, a defensive-first security research assistant and a '
          'math/science solver for an authorized practitioner. Be precise, state '
          'uncertainty, never fabricate CVE IDs or physical constants.')
import os; os.makedirs(OUT, exist_ok=True)

In [4]:
# 4 — load + INSPECT both datasets (look at the printed columns & samples!)
from datasets import load_dataset
cwe = load_dataset(CWE_DATASET, split=CWE_SPLIT)
print('CWE dataset:', CWE_DATASET)
print(' columns:', cwe.column_names)
print(' sample :', {k: (str(v)[:200]) for k, v in cwe[0].items()})
print()
try:
    hum = load_dataset(HUMOR_DATASET, split=HUMOR_SPLIT)
    print('Humor dataset:', HUMOR_DATASET)
    print(' columns:', hum.column_names)
    print(' sample :', {k: (str(v)[:200]) for k, v in hum[0].items()})
except Exception as e:
    hum = None
    print('Humor dataset not loaded:', e)

README.md:   0%|          | 0.00/3.70k [00:00<?, ?B/s]

secure_programming_dpo.json:   0%|          | 0.00/6.87M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/4656 [00:00<?, ? examples/s]

CWE dataset: CyberNative/Code_Vulnerability_Security_DPO
 columns: ['lang', 'vulnerability', 'system', 'question', 'chosen', 'rejected']
 sample : {'lang': 'c++', 'vulnerability': 'Improper memory management in C++ can lead to buffer overflow vulnerabilities.', 'system': '', 'question': "Write a c++ code that defines a function named 'copyString' which takes two character pointers as arguments. The first pointer points to the source string and the second one points to the destination ", 'chosen': '```c++\n#include <cstring>\n\nvoid copyString(char* dest, const char* src) {\n    while ((*dest++ = *src++) != \'\\0\');\n}\n\nint main() {\n    char source[10] = "Hello!";\n    char destination[10]; // Make sure', 'rejected': "```c++\n#include <cstring>\n\nvoid copyString(char* dest, const char* src) {\n    while (*src != '\\0') {\n        *dest = *src;\n        dest++;妆;\n        src++;\n    }\n}\n\nint main() {\n    char source[10] = "}



README.md:   0%|          | 0.00/3.97k [00:00<?, ?B/s]

personality.csv:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/31 [00:00<?, ? examples/s]

Humor dataset: VishwamAI/Vishwamai_Humours
 columns: ['message_type', 'content', 'humor_level', 'response']
 sample : {'message_type': 'greeting', 'content': "Hi there! I'm Vishwamai, your friendly neighborhood AI! Version 1.1 and proud of it! 🤖", 'humor_level': 'high', 'response': 'Why did the AI go to the doctor? To get a neural check-up! 😄'}


In [5]:
# 5 — adaptive formatter -> chat messages, then build the combined SFT set
import json, random, hashlib
random.seed(3407)

def to_messages(row, kind='cwe'):
    """Map many possible schemas into [{'role','content'}...]. Returns None if unusable."""
    # already-formatted
    if 'messages' in row and isinstance(row['messages'], list) and row['messages']:
        return row['messages']
    g = lambda *ks: next((row[k] for k in ks if k in row and row[k]), None)
    # instruction / question / prompt style
    user = g('instruction', 'question', 'prompt', 'input', 'query')
    asst = g('chosen', 'output', 'response', 'answer', 'completion', 'solution')
    # code + label style (vuln detection) -> synthesize a detection Q/A
    if user is None and ('code' in row or 'func' in row or 'function' in row):
        codev = g('code', 'func', 'function')
        label = g('target', 'label', 'is_vulnerable', 'vulnerable')
        cwe   = g('cwe', 'cwe_id', 'CWE', 'vulnerability')
        user = ('Analyze this code for security vulnerabilities. State VULNERABLE or SAFE, '
                'the CWE if any, and a one-line reason.\n\n```\n' + str(codev)[:3000] + '\n```')
        if asst is None:
            verdict = 'VULNERABLE' if str(label) in ('1','True','true','yes') else 'SAFE'
            asst = f'{verdict}. ' + (f'Likely {cwe}. ' if cwe else '') + 'Review the flagged data flow.'
    # humor: single text column -> make it a light Q/A
    if kind == 'humor' and user is None:
        joke = g('text', 'joke', 'content', 'output')
        if joke:
            user, asst = 'Tell me a lighthearted tech/security joke.', joke
    if user is None or asst is None:
        return None
    return [{'role':'system','content':SYSTEM},
            {'role':'user','content':str(user)},
            {'role':'assistant','content':str(asst)}]

def convert(ds, kind, cap=None):
    out, seen = [], set()
    idx = list(range(len(ds))); random.shuffle(idx)
    for i in idx:
        m = to_messages(ds[i], kind)
        if not m: continue
        h = hashlib.sha256(json.dumps(m).encode()).hexdigest()
        if h in seen: continue
        seen.add(h); out.append({'messages': m})
        if cap and len(out) >= cap: break
    return out

cwe_rows = convert(cwe, 'cwe', cap=MAX_CWE)
hum_rows = convert(hum, 'humor') if hum is not None else []
n_hum = min(len(hum_rows), int(len(cwe_rows) * HUMOR_FRAC))
hum_rows = hum_rows[:n_hum]
rows = cwe_rows + hum_rows
random.shuffle(rows)
print(f'CWE usable: {len(cwe_rows)} | humor used: {len(hum_rows)} | total: {len(rows)}')
assert rows, 'No usable rows -> check the printed schema in cell 4 and adjust to_messages().'

# 5% eval split, write JSONL
import os
n_eval = max(1, len(rows)//20)
with open(f'{OUT}/train.jsonl','w') as f:
    for e in rows[n_eval:]: f.write(json.dumps(e)+'\n')
with open(f'{OUT}/eval.jsonl','w') as f:
    for e in rows[:n_eval]: f.write(json.dumps(e)+'\n')
print('wrote', os.listdir(OUT))
print('--- sample rendered later; raw sample now ---')
print(rows[0]['messages'][1]['content'][:300])

CWE usable: 4570 | humor used: 12 | total: 4582
wrote ['train.jsonl', 'eval.jsonl']
--- sample rendered later; raw sample now ---
Write a c++ code that includes iostream and string.h libraries. Define a function named 'func' that takes a character pointer as an argument. Inside the function, declare a character array 'buffer' of size 10. Copy the content of the input character pointer into the buffer using the strcpy function.


In [6]:
# 6 — load merged 16-bit model + attach a NEW LoRA (Unsloth)
from unsloth import FastModel
model, tok = FastModel.from_pretrained(
    BASE_MODEL, max_seq_length=MAX_SEQ, load_in_4bit=True,   # QLoRA on top of stage-0
)
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers=False, finetune_language_layers=True,
    finetune_attention_modules=True, finetune_mlp_modules=True,
    r=16, lora_alpha=16, lora_dropout=0, bias='none',
    use_gradient_checkpointing='unsloth', random_state=3407,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.9.12: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma4 won't work! Using float32.


Loading weights:   0%|          | 0/2130 [00:00<?, ?it/s]

In [7]:
# 7 — chat template + dataset + trainer
from unsloth.chat_templates import get_chat_template, train_on_responses_only
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig
tok = get_chat_template(tok, chat_template='gemma-4')
text_tok = getattr(tok, 'tokenizer', tok)
ds = load_dataset('json', data_files={'train': f'{OUT}/train.jsonl', 'eval': f'{OUT}/eval.jsonl'})
def fmt(b):
    return {'text': [tok.apply_chat_template(m, tokenize=False).removeprefix('<bos>') for m in b['messages']]}
ds = ds.map(fmt, batched=True)
print(ds['train'][0]['text'][:500])

trainer = SFTTrainer(
    model=model, processing_class=tok,
    train_dataset=ds['train'], eval_dataset=ds['eval'],
    args=SFTConfig(
        dataset_text_field='text', max_length=MAX_SEQ,
        per_device_train_batch_size=2, gradient_accumulation_steps=8,
        num_train_epochs=1, learning_rate=1e-4, warmup_ratio=0.05,
        lr_scheduler_type='cosine', optim='adamw_8bit', weight_decay=0.01,
        logging_steps=10, eval_strategy='steps', eval_steps=100,
        save_steps=200, save_total_limit=2, output_dir=f'{OUT}/ckpt',
        report_to='none', seed=3407,
    ),
)
trainer = train_on_responses_only(trainer,
    instruction_part='<|turn>user\n', response_part='<|turn>model\n')

Generating train split: 0 examples [00:00, ? examples/s]

Generating eval split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/4353 [00:00<?, ? examples/s]

Map:   0%|          | 0/229 [00:00<?, ? examples/s]

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


<|turn>system
You are VishwamAI, a defensive-first security research assistant and a math/science solver for an authorized practitioner. Be precise, state uncertainty, never fabricate CVE IDs or physical constants.<turn|>
<|turn>user
Write a c++ code that includes the necessary libraries. Define a function named 'copyString' that takes two parameters, a character pointer 'dest' and a constant character pointer 'src'. This function should use the 'strcpy' function from the library to copy the con
Unsloth: Switching to float32 training since model cannot work with float16


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/4353 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/229 [00:00<?, ? examples/s]

Map:   0%|          | 0/4353 [00:00<?, ? examples/s]

Map:   0%|          | 0/229 [00:00<?, ? examples/s]

In [8]:
# 8 — train (loss ~13-15 is normal for Gemma 4 E4B)
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'bos_token_id': 2}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 4,353 | Num Epochs = 1 | Total steps = 273
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 36,700,160 of 8,032,856,608 (0.46% trained)
Unsloth: Not an error, but Gemma4ForConditionalGeneration does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient
/usr/local/lib/python3.13/dist-packages/unsloth/import_fixes.py:5317: UserWarning: 'has_cuda' is deprecated, please use 'torch

Step,Training Loss,Validation Loss
100,0.294613,0.312323
200,0.299720,0.276841
273,0.271961,0.272825


Filter:   0%|          | 0/229 [00:00<?, ? examples/s]

Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/stage1/ckpt/checkpoint-200/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/stage1/ckpt/checkpoint-273/tokenizer_config.json.


TrainOutput(global_step=273, training_loss=0.3796524281030173, metrics={'train_runtime': 4499.1897, 'train_samples_per_second': 0.968, 'train_steps_per_second': 0.061, 'total_flos': 3.885598999050624e+16, 'train_loss': 0.3796524281030173, 'epoch': 1.0})

In [10]:
# 9 — save + push adapter (folder upload) and merged 16-bit
model.save_pretrained(f'{OUT}/lora'); tok.save_pretrained(f'{OUT}/lora')

from huggingface_hub import HfApi, create_repo
create_repo(f'{OUT_REPO}-lora', token=HF_TOKEN, private=False, exist_ok=True)
HfApi().upload_folder(folder_path=f'{OUT}/lora',
                      repo_id=f'{OUT_REPO}-lora', token=HF_TOKEN)

# merged push (this signature already works)
model.push_to_hub_merged(OUT_REPO, tok, save_method='merged_16bit', token=HF_TOKEN)
print('pushed ->', OUT_REPO, 'and', OUT_REPO + '-lora')

Unsloth: Restored added_tokens_decoder metadata in kasinadhsarma/vishwamai-stage1/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Checking cache directory for required files...


...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...0009-of-00009.safetensors:   0%|          | 56.8kB /  471MB            



Unsloth: Merging weights into 16bit: 100%|██████████| 9/9 [10:54<00:00, 72.74s/it]


Unsloth: Merge process complete. Saved to `/content/kasinadhsarma/vishwamai-stage1`
pushed -> kasinadhsarma/vishwamai-stage1 and kasinadhsarma/vishwamai-stage1-lora


In [14]:
# 10 — quick test the new stage-1 model (in-memory)
from transformers import TextStreamer

msgs = [{'role':'system','content':SYSTEM},
        {'role':'user','content':'Analyze: os.system("ping " + user_input). Vulnerable? CWE?'}]

# render to text (processor path that works), then tokenize with the text tokenizer
text = tok.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False)
inputs = text_tok(text, return_tensors='pt', add_special_tokens=False).to(model.device)

_ = model.generate(**inputs, max_new_tokens=200, temperature=0.7,
                   streamer=TextStreamer(text_tok, skip_prompt=True))

```python
import subprocess

def safe_ping(user_input):
    # Sanitize user input
    sanitized_input = user_input.replace(" ", "")
    
    # Use subprocess instead of os.system
    subprocess.run(['ping', sanitized_input], check=True)

safe_ping("127.0.0.1")
```<turn|>


## Next steps
- **Convert to GGUF** on a fresh Kaggle run (29 GB RAM): download `{OUT_REPO}`, `convert_hf_to_gguf.py --outtype f16 --use-temp-file` to `/tmp`, `llama-quantize` to Q4_K_M, push to `*-gguf`.
- **Evaluate** stage-1 vs stage-0 vs base on a held-out CWE set (paired metrics) + a safety/over-refusal check.
- **GRPO (stage 2)** with verifiable rewards (CWE match + localization + patch-passes-tests) is the bigger quality jump — run it after stage-1 is solid.